# CME Futures: Sequence Models

This notebook evaluates the declared NLinear and LSTM sequence configurations. Each input window
contains observations from one product and ends before its prediction timestamp. Purge gaps and
fold boundaries prevent a sequence from crossing into another validation interval, and hidden
state does not pass between products or folds.

Every declared epoch checkpoint is published with fitted weights and exact chronological
eligibility. MC dropout is not an undeclared side experiment. Configuration selection remains the
validation backtest decision in `13_backtest`.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：序列模型

本 Notebook 評估已宣告的 NLinear 與 LSTM 序列模型設定。每個輸入視窗只包含單一商品的觀測值，且結束時間早於預測時間戳記。淨空間隔與資料折邊界防止序列跨入其他驗證區間；隱藏狀態也不會在商品或資料折之間傳遞。

每個已宣告訓練週期檢查點，都會連同擬合權重及確切的時間順序合格條件一併發布。蒙地卡羅 Dropout（MC dropout）並非未宣告的附帶實驗。設定選擇仍由 `13_backtest` 的驗證回測決定。

先決條件：`03_financial_features`、`04_model_based_features` 與 `05_evaluation`。
<!-- ML4T_ZH_END c0000-p000 -->


## What a sequence model reads that the other families do not

Every family up to this point saw one row per product per decision: a vector of features
describing that product at that moment. Anything about how it got there had to be engineered
into a column - a 21-session volatility, a momentum composite, a carry z-score against a
rolling window. The model saw the summary, never the path.

A sequence model reads the path. Its input is a window of consecutive observations for one
product, and the architecture is built to make use of their order. The claim being tested is
that the shape of recent history carries information that no fixed set of summary statistics
captured - that a product whose carry rose steadily to its current level differs from one that
spiked and fell back, even where both end at the same value with the same 21-session
volatility.

For thirty futures products the windows are also the scarcest data in the case study. A
feature-row model gets one training example per product per session; a sequence model needs a
whole window per example, so the same history yields fewer independent examples and they
overlap heavily with each other. That is the structural reason to expect these models to
struggle here relative to a benchmark with millions of series, and it is worth holding
alongside whatever the backtest reports.

### Two architectures, and why both

**LSTM** processes the window one step at a time, carrying a hidden state forward and learning
what to keep and what to forget. It is the general answer, and its generality is the cost: it
has many parameters, it trains slowly, and on a short noisy series it has ample capacity to
memorize.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 序列模型讀到哪些其他模型類別讀不到的資訊

到目前為止，每種模型類別在每次決策時，只看到每項商品的一列資料：描述該商品當下狀態的特徵向量。商品如何走到該狀態，都必須先經特徵工程濃縮為欄位，例如 21 個交易日的波動率、動能綜合指標，或相對滾動視窗計算的持有收益 z 分數。模型只看到摘要，從未看到路徑。

序列模型讀取路徑。輸入是單一商品連續觀測值的視窗，模型架構則設計來利用其順序。要檢驗的主張是：近期歷史的形狀，包含固定一組摘要統計量未捕捉到的資訊。例如，某商品的持有收益穩定上升至目前水準，與另一商品先急升再回落，即使兩者最後數值及 21 個交易日波動率相同，仍有所差異。

對這 30 項期貨商品，視窗也是本案例研究中最稀缺的資料。使用特徵資料列的模型，每項商品每個交易日都有一筆訓練樣本；序列模型每筆樣本卻需要一整個視窗，因此相同歷史資料產生的獨立樣本較少，視窗間也大量重疊。相較於擁有數百萬條序列的基準資料，這是預期此處模型表現較困難的結構性原因；解讀回測結果時也應一併考慮。

### 兩種架構及同時使用的原因

**LSTM** 逐步處理視窗，向前傳遞隱藏狀態，並學習要保留與遺忘的資訊。它是一般性的解法，而一般性也有代價：參數多、訓練慢，在短而雜訊高的序列上，擁有充足容量可記住樣本。
<!-- ML4T_ZH_END c0001-p000 -->

**NLinear** is close to the opposite. It is a linear map from the window to the forecast, with
a normalization step that subtracts the window's last value before the map and adds it back
afterwards. That subtraction is the whole idea: it makes the model predict the *change* from
where the series currently sits rather than the level, which removes the drift that otherwise
dominates a naive fit.

It is here because a body of recent work found that simple linear baselines matched or beat
elaborate sequence architectures on many forecasting benchmarks once evaluated carefully - a
finding that survived enough scrutiny to be worth designing around. Running both is what turns
"the sophisticated model should win" into something this case study measures rather than
assumes.

## Where a sequence model can leak, and what stops it

A window is a span of time rather than a point, which gives leakage more places to enter than
the other families have.


<!-- ML4T_ZH_BEGIN c0001-p001 -->
**NLinear** 幾乎是相反的做法。它以線性映射將視窗轉成預測值；映射前先減去視窗最後一個數值，映射後再將該值加回。減去最後數值正是其核心：模型因此預測序列相對目前位置的*變化*，而非數值水準，排除原本會主導簡單擬合的漂移。

納入它的原因是，近年的一系列研究發現，在仔細評估後，簡單的線性基準在許多預測基準測試上，能追平甚至超越複雜的序列架構。這項發現經過足夠檢驗，值得納入設計。同時執行兩者，才能讓「複雜模型應該勝出」成為本案例研究實際衡量的問題，而非預設結論。

## 序列模型可能在哪裡洩漏資訊，以及如何防止

視窗涵蓋一段時間，而非單一時點，因此比其他模型類別多了幾個資訊洩漏的入口。
<!-- ML4T_ZH_END c0001-p001 -->

- **Each window ends before its prediction timestamp.** The last observation a window contains
  is strictly earlier than the moment being predicted, so a forecast never reads the bar it is
  forecasting.
- **Purge gaps and fold boundaries stop a window crossing into another interval.** Without them
  a window ending just after a fold boundary would extend back across it, and validation rows
  would be predicted from a window overlapping the training period. The failure would be
  invisible in the output: the prediction is dated correctly and the returns are real.
- **Hidden state does not pass between products or folds.** An LSTM's state accumulates
  whatever it has seen, so carrying it across a boundary carries information across that
  boundary too - and unlike a feature column, the state never appears in any frame, so nothing
  downstream could detect it.

The three are separate mechanisms rather than one guarantee stated three times, which is why
they are enforced separately rather than by a single check on the output.

<!-- ML4T_ZH_BEGIN c0001-p002 -->
- **每個視窗都在預測時間戳記之前結束。**視窗最後一筆觀測值嚴格早於預測時刻，因此預測絕不會讀到正在預測的價格資料。
- **淨空間隔與資料折邊界阻止視窗跨入另一區間。**若沒有這些限制，剛好在資料折邊界後結束的視窗會向前跨越邊界，使驗證資料列的預測使用與訓練期間重疊的視窗。輸出上看不出問題：預測日期正確，報酬也是真實的。
- **隱藏狀態不會在商品或資料折之間傳遞。**LSTM 的狀態會累積已見資訊，因此跨邊界傳遞狀態，也會把資訊帶過邊界。與特徵欄位不同，狀態不會出現在任何資料框中，後續程序也無從偵測。

這三項是不同機制，不是以三種說法重複同一項保證，因此會分別強制執行，而非只在輸出上進行一次檢查。
<!-- ML4T_ZH_END c0001-p002 -->


In [1]:
"""Fit the declared CME futures sequence-model population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)
from case_studies.research import population_supersedes

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# The population hash this run replaces, read from the registry and set by a person. A
# first population takes None; a re-run whose membership has changed is refused without
# the hash it supersedes, and the refusal names the value required.
SUPERSEDES_POPULATION: str | None = "8c2c87299a47"
# The device to fit on. Empty means the device this population was published on.
DEVICE: str = ""
# The population this run publishes into. Empty publishes the canonical one, which a run
# on another device may not do.
POPULATION_NAME: str = ""

## Declared requests

The request rows identify architecture, label, and published configuration. Sequence length,
checkpoint schedule, seed, gap policy, and device enter the resolved computation identity.

**The device is declared here rather than inherited.** With no override the shared sequence
adapter falls back to a literal `"cuda"` written in `case_studies/utils/deep_learning.py`, and
resolving the request raises `CUDA was requested for sequence training, but CUDA is unavailable`
rather than quietly moving the fit to the CPU. That refusal comes from resolving the request, so
it arrives before any fitting starts. Stating it in the request puts that requirement where a
reader meets it instead of two layers below. The resolved specification hash is the same with the
override as without, so this names what the published run already did.

The device is part of what the fitted model is, not a note beside it: the same architecture
trained on a GPU and on a CPU accumulates its sums in different orders and reaches different
weights. `PUBLISHED_DEVICE` is the device this population was fitted on, and the canonical
population accepts no other. A reader without an NVIDIA card sets `DEVICE="cpu"` and passes a
`POPULATION_NAME` to fit the same grid into a population of its own, which the backtest does
not read.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 已宣告的訓練請求

各請求資料列指出架構、標籤及已發布設定。序列長度、檢查點排程、種子、間隔政策及運算裝置，都納入解析後的運算識別資訊。

**運算裝置在此明確宣告，而非繼承預設值。**若未覆寫，共用序列轉接器會退回 `"cuda"` 這個字面值；它寫在 `case_studies/utils/deep_learning.py` 中，而解析請求時會引發 `CUDA was requested for sequence training, but CUDA is unavailable`，不會悄悄改在 CPU 上擬合。這項拒絕發生於解析請求時，因此在任何擬合開始前就會出現。把要求寫在請求中，讀者便能直接看到，而不必往下追查兩層。無論是否覆寫，解析後的規格雜湊都相同，因此這只是明確指出已發布執行原本使用的裝置。

運算裝置是擬合後模型的一部分，而非旁邊的備註：相同架構在 GPU 與 CPU 上訓練，會以不同順序累加數值並得到不同權重。此模型群組是在 `PUBLISHED_DEVICE` 上擬合，正式群組不接受其他裝置。沒有 NVIDIA 顯示卡的讀者，可設定 `DEVICE="cpu"` 並傳入 `POPULATION_NAME`，將同一網格擬合至自己的模型群組；回測不會讀取該群組。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
PUBLISHED_DEVICE = "cuda"
device = DEVICE or PUBLISHED_DEVICE
if device != PUBLISHED_DEVICE and not POPULATION_NAME:
    raise ValueError(
        f"this run fits on device {device!r}, which is not the {PUBLISHED_DEVICE!r} this "
        f"population was published on, so it cannot publish the canonical population; pass "
        f"POPULATION_NAME to give it its own"
    )

population_name = POPULATION_NAME or "cme_futures-deep_learning-validation-v1"

study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog("deep_learning", labels=ALL_LABELS)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""regression""",71,30,37782,5,2019-01-03 00:00:00,2023-11-29 00:00:00,20,"""canonical""","""79752926dbf0"""
"""deep_learning""","""fwd_ret_21d""","""nlinear""","""regression""",71,30,37782,5,2019-01-03 00:00:00,2023-11-29 00:00:00,20,"""canonical""","""8c2861975ad2"""
"""deep_learning""","""fwd_ret_5d""","""lstm_h64""","""regression""",71,30,38262,5,2019-01-03 00:00:00,2023-12-21 00:00:00,20,"""canonical""","""9521376e1d23"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""regression""",71,30,38262,5,2019-01-03 00:00:00,2023-12-21 00:00:00,20,"""canonical""","""0acd75b9bb72"""


## Execute and validate

The shared sequence adapter owns window construction, checkpoint reload, prediction coverage, and
restart. A failed configuration cannot remove itself from the population snapshot.

**"Cannot remove itself" is the load-bearing clause.** The natural way to write a sweep is to
catch a failure, log it, and carry on with what worked - which produces a population defined
by what happened to train rather than by what was declared. The leaderboard still looks
sensible, and the configuration that failed is indistinguishable from one that was never
requested. Sequence models make this more likely than the other families do, because they are
the ones that run out of memory or fail to converge on a thin product.

### What MC dropout is, and why it is declared rather than switched on

Dropout during training randomly disables units so the network cannot rely on any one path.
**MC dropout** leaves it enabled at prediction time and runs the forward pass several times, so
each pass gives a slightly different answer and their spread estimates the model's uncertainty
about that prediction.

That is a useful quantity - it is what an allocator sizing inversely to uncertainty would want
- but it changes what the model outputs. A prediction averaged over stochastic passes is not
the same number as the deterministic one, and a run that quietly enabled it would publish
different values under the same configuration name. So it is part of the declared
configuration and enters the identity, which is what the header means by "not an undeclared
side experiment": either the population says these predictions are MC-dropout predictions, or
they are not, and no run gets to decide that on its own.

### Why checkpoints are published rather than chosen


<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行與驗證

共用序列轉接器負責建構視窗、重新載入檢查點、確保預測涵蓋範圍及重新啟動。失敗的設定不能自行從模型群組快照中消失。

**「不能自行消失」是這裡的關鍵條件。**撰寫一系列設定的訓練程序時，直覺做法是捕捉失敗、記錄後繼續處理成功的設定；但這會讓模型群組由實際成功訓練的內容決定，而非由事先宣告的內容決定。排行榜看起來仍合理，失敗設定也與從未提出訓練請求的設定無法區分。序列模型比其他模型類別更可能出現這種狀況，因為它們較容易耗盡記憶體，或在樣本稀少的商品上無法收斂。

### MC dropout 是什麼，以及為何須先宣告而非臨時啟用

訓練時的 Dropout 會隨機停用部分單元，使網路無法依賴單一路徑。**MC dropout** 在預測時仍保持啟用，並多次執行前向傳遞；每次結果略有不同，其分散程度可估計模型對該次預測的不確定性。

這是有用的量，按不確定性反比配置部位的配置器正需要它；但它會改變模型的輸出。多次隨機前向傳遞的平均預測，並不等於確定性預測。若某次執行悄悄啟用它，就會在相同設定名稱下發布不同數值。因此它屬於已宣告設定的一部分，並納入識別資訊。這正是開頭所說「並非未宣告的附帶實驗」之意：模型群組要嘛聲明這些預測使用 MC dropout，要嘛未使用；任何一次執行都不能自行決定。

### 為何發布檢查點，而非事先選定
<!-- ML4T_ZH_END c0007-p000 -->

As in `08_tabular_dl`: a neural fit is a trajectory, and choosing the best epoch by validation
performance before reporting that model's validation performance is selection inside the
number being reported. Every declared checkpoint becomes a candidate row and `13_backtest`
selects among them on Sharpe, so the choice sits in the same funnel and the same trial count
as everything else.

<!-- ML4T_ZH_BEGIN c0007-p001 -->
如同 `08_tabular_dl`：神經網路的擬合是一條軌跡；若先依驗證表現選出最佳訓練週期，再報告該模型的驗證表現，就等於在所報數值內部進行選擇。每個已宣告檢查點都成為候選資料列，`13_backtest` 再依夏普比率選擇，因此這項決策會與其他設定一起進入相同選擇流程及試驗次數計算。
<!-- ML4T_ZH_END c0007-p001 -->


In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name=population_name,
        resolved_requests=resolved,
        supersedes=population_supersedes(
            study,
            name=population_name,
            declared=SUPERSEDES_POPULATION,
        ),
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "config_name", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("sequence execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",5,"""canonical""",true,"""79752926dbf0""","""31e6a5c64228"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",10,"""canonical""",true,"""79752926dbf0""","""243d0d3c7165"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",15,"""canonical""",true,"""79752926dbf0""","""2969c6f856d3"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",20,"""canonical""",true,"""79752926dbf0""","""74c22a14939b"""
"""deep_learning""","""fwd_ret_21d""","""lstm_h64""","""epoch""",25,"""canonical""",true,"""79752926dbf0""","""554aa3766fca"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",80,"""canonical""",true,"""0acd75b9bb72""","""15ab04e27327"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",85,"""canonical""",true,"""0acd75b9bb72""","""9720a7aef984"""
"""deep_learning""","""fwd_ret_5d""","""nlinear""","""epoch""",90,"""canonical""",true,"""0acd75b9bb72""","""8c56261dff93"""
